# Starting with rag
## Requirements :
    -langchain-community
    -PyPDF
    -PymuPDF


### loading documents

In [2]:
# Text Loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../doc_files/notes.txt")
content = loader.load()
print(content)


C:\Users\Vivek\AppData\Local\Temp\ipykernel_16616\1387347472.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../doc_files/notes.txt'}, page_content='This is a simple text file content.')]


In [ ]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

dirLoader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema": ".", "text_content": False},
    show_progress=False
)
content = dirLoader.load()
content

[Document(metadata={'source': 'C:\\Users\\Vivek\\Desktop\\agentic_ai\\doc_files\\config.json', 'seq_num': 1}, page_content='{"setting": "enabled", "version": 1}')]

In [ ]:
# Loading Pdf File
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    loader_kwargs={"extract_images":True, "extract_tables": "markdown"},
    use_multithreading=True
)
contents = dir_loader.load()
contents

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'file_path': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-08-03T16:14:20+05:30', 'trapped': '', 'modDate': "D:20260803161420+05'30'", 'creationDate': "D:20260803161420+05'30'", 'page': 0}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands-on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD-driven applications with structured data \nmanagement, exception handling, and input validation. Strong fou

In [4]:
# Create dummy files and write the some content 

import os

storage = {
    "notes.txt": "This is a simple text file content.",
    "config.json": '{"setting": "enabled", "version": 1.0}',
    "script.py": "print('Hello from the script!')",
}

for file, content in storage.items():
    with open(f"../doc_files/{file}", "w", encoding="utf-8") as f:
        f.write(content)


print("content Written Successfully...")

content Written Successfully...


# RAG Pipeline (From Indexing to Vector Db pipleine)
### Requirements: 
    -langchain-community(PyPDFLoader and PyMuPDF)
    -langchain.textsplitter (RecurisveCharacterTextSplitter)
    pathlib

In [12]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [30]:
# create a function that loads all the pdf file in the dir and returns the whole documents by adding corresponding metadata fields like file_name and filetype
def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file.stem}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")

Processing :SowmyaG_Resume
Loaded 1 pages
Processing :VivekanandNaikCV
Loaded 1 pages
Processing :Vivekanand_Naik_Resume
Loaded 1 pages


In [32]:
# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    for i in text_splitter.split_documents(docs):
        print(i)
    return True
split_doc(all_docs)

page_content='SOWMYA G 
Software Developer | Python & Django Developer | Backend Development 
+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G 
SUMMARY 
Computer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications 
using object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data 
management, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version 
control through Git and GitHub. 
TECHNICAL SKILLS 
Languages: Python, Java 
Web & Frameworks: HTML5, CSS3, Django 
Databases: SQL (fundamentals) 
Tools & Platforms: Git, GitHub, VS Code 
Core Concepts: Data Structures & Algorithms, Object-Oriented Programming (OOP), File Handling, Exception Handling, Problem 
Solving 
EXPERIENCE 
Python and Web Development Trainee — Tap Academy May 2025 – June 2025' metadata={'produce

True